# How to use the dataset: time series
In this notebook, we show examples of usage for dataset structured as time series. The objective is to detect attack sequences by detecting one of the attack datapoint that compose it as soon as possible. The ideal to have a early detection of attacks would be to detect the first data point in a sequence (as it is the one that first occur in time)  
- We prepare attacks and normal behaviour sequence of data points.
-

In [37]:
import pandas as pd
import numpy as np

In [38]:
features = np.load('./data/usable_features.npy')

In [39]:
features = list(features)

In [40]:
features.append('attack')
features.append('timestamp')

In [41]:
import pandas as pd

# 只讀取第一列，不指定 usecols，看看裡面到底有什麼
df_preview = pd.read_csv('./Untitled Folder/ROSPaCe_complete.csv', nrows=1)

print("--- 檔案中的欄位名稱 ---")
print(df_preview.columns.tolist())

print("\n--- 檔案中的欄位數量 ---")
print(len(df_preview.columns))

# 檢查你的 features 是否真的在裡面
# 假設你的 features 變數已經載入
# missing_cols = [c for c in features if c not in df_preview.columns]
# print(f"\n--- 仍然缺失的欄位 ({len(missing_cols)} 個) ---")
# print(missing_cols[:10]) # 只印出前 10 個給你看

--- 檔案中的欄位名稱 ---
['timestamp', 'layers.frame.frame.time', 'layers.frame.frame.time_delta', 'layers.frame.frame.time_delta_displayed', 'layers.frame.frame.time_relative', 'layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.protocols', 'layers.sll.sll.pkttype', 'layers.sll.sll.hatype', 'layers.sll.sll.src.eth', 'layers.sll.sll.unused', 'layers.sll.sll.etype', 'layers.ip.ip.version', 'layers.ip.ip.hdr_len', 'layers.ip.ip.dsfield', 'layers.ip.ip.dsfield_tree.ip.dsfield.dscp', 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn', 'layers.ip.ip.len', 'layers.ip.ip.id', 'layers.ip.ip.flags', 'layers.ip.ip.flags_tree.ip.flags.rb', 'layers.ip.ip.flags_tree.ip.flags.df', 'layers.ip.ip.flags_tree.ip.flags.mf', 'layers.ip.ip.flags_tree.ip.frag_offset', 'layers.ip.ip.ttl', 'layers.ip.ip.proto', 'layers.ip.ip.checksum', 'layers.ip.ip.checksum.status', 'layers.ip.ip.src', 'layers.ip.ip.addr', 'layers.ip.ip.src_host', 'layers.ip.ip.host', 'layers.ip.ip.dst', 

In [42]:
df = pd.read_csv('./Untitled Folder/ROSPaCe_complete.csv', usecols=features)

/tmp/ipykernel_555/1660289087.py:1: DtypeWarning: Columns (21,28,55,65,96,100,476,479,480) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./Untitled Folder/ROSPaCe_complete.csv', usecols=features)


In [43]:
print(df.shape)

(30247050, 62)


## Prepare the data for ML
In this section, we prepare the data to be processed by ML algorithms. In particular, we perform the following steps:
- <strong>Sort by timestamp </strong>: we want to compose time-ordered sequence of normal and attack data points. 
- <strong>Convert Label to Numeric</strong>: We substitute label values with numeric values. Then, we create two versions of the dataset: with  binary labels (attack, normal), and with multiple labels (one label for each attack).
- <strong>Convert String To Numeric</strong>: we convert the string values to numbers using categorical encoding. This technique assigns a unique number to any unique string values of a feature.
- <strong>Create sequences of normal and attack istances: we alterante sequences of normal and attack data point</strong>:
- <strong>Split the dataset in Training and Test sets</strong>: after removing labels and timestamps columns, we split the dataframe in training and test sets with a 60/40 split.


In [44]:
df['timestamp'] = pd.to_datetime(df['timestamp'],
               format='%Y-%m-%d %H:%M:%S.%f')

In [45]:
df = df.sort_values('timestamp')

In [46]:
print(df['timestamp'])

0          2023-03-16 14:22:23.903192576
1          2023-03-16 14:22:23.903765248
2          2023-03-16 14:22:23.904402432
3          2023-03-16 14:22:23.904744448
4          2023-03-16 14:22:23.905764096
                        ...             
30247045   2023-06-16 19:58:12.266677760
30247046   2023-06-16 19:58:12.267390208
30247047   2023-06-16 19:58:12.267472640
30247048   2023-06-16 19:58:12.267490048
30247049   2023-06-16 19:58:12.309100288
Name: timestamp, Length: 30247050, dtype: datetime64[ns]


In [47]:
def convert_dtype(x):
    if not x:
        return ''
    try:
        return str(x)   
    except:        
        return ''
    
def convert_hex(x):
    if not x:
        return 0
    try:
        return literal_eval(x)
    except:        
        return 0

In [48]:
#Indexes of the columns to be converted
to_convert = [11,18,32,42]

In [49]:
#Convert
for i in to_convert:
    df[df.columns[i]] = df[df.columns[i]].apply(lambda x: convert_dtype(x))

### Convert Label columns to Numeric values
We substitute label values with numeric values. We create two version of the dataset:
- binary classification
- multiple label classification

In [50]:
mutli_atk_label = df['attack']

df['attack'] = df['attack'].replace('metasploit SYN flood', 1) 
df['attack'] = df['attack'].replace('nmap discovery', 2)
df['attack'] = df['attack'].replace('nmap SYN flood', 3) 
df['attack'] = df['attack'].replace('ros2 node crashing', 4)
df['attack'] = df['attack'].replace('ros2 reconnaissance', 5)
df['attack'] = df['attack'].replace('ros2 reflection', 6)
df['attack'] = df['attack'].replace('observe', 0)

df['attack'] = pd.to_numeric(df['attack'])

df['attack'].unique(), df['attack'].nunique()

/tmp/ipykernel_555/1465565132.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['attack'] = df['attack'].replace('observe', 0)


(array([0, 2, 5, 6, 4, 3, 1]), 7)

In [51]:
print(df['attack'].value_counts())

attack
1    14696890
3     7993846
0     6645324
2      500000
5      401494
4        5743
6        3753
Name: count, dtype: int64


In [52]:
df['attack'] = df['attack'].replace(2, 1)
df['attack'] = df['attack'].replace(3, 1) 
df['attack'] = df['attack'].replace(4, 1)
df['attack'] = df['attack'].replace(5, 1)
df['attack'] = df['attack'].replace(6, 1)

In [ ]:
df.to_csv('./data/bin_final_reduced.csv')

In [ ]:
df = pd.read_csv('./data/bin_final_reduced.csv')

### Convert String to Numeric

In [ ]:
list_column_string=df.select_dtypes(exclude=[np.number])

for i in list_column_string:
    if i != 'timestamp':
        df[i] = pd.Categorical(df[i])

In [ ]:
for i in list_column_string:
    if i != 'timestamp':
        df[i] = df[i].cat.codes

## Create blocks of Attack and Normal datapoints
- We split the dataframe to have a list of blocks composed by a sequence of data points. Each block contains exclusively attack or normal datapoints.

In [ ]:
#Find consecutive sequences of the same value
sequences = []
sequence_value = None
sequence_start_index = None

for index, row in df.iterrows():
    if row['attack'] == sequence_value:
        continue  # Continue the current sequence
    else:
        if sequence_value is not None:
            sequences.append((sequence_value, sequence_start_index, index - 1))
        sequence_value = row['attack']
        sequence_start_index = index

# Append the last sequence
if sequence_value is not None:
    sequences.append((sequence_value, sequence_start_index, len(df) - 1))

# Retrieve indices for each sequence
sequence_indices = []
for seq_value, start_idx, end_idx in sequences:
    indices = df.index[start_idx:end_idx + 1].tolist()
    sequence_indices.append((seq_value, indices))

In [ ]:
sequence_indices = np.array(sequence_indices,dtype=object)

In [ ]:
np.save('./data/sequences_index.npy', sequence_indices)

In [ ]:
sequence_indices = np.load('./data/sequences_index.npy', allow_pickle=True) #sequences_index_5ml.npy

In [ ]:
print(sequence_indices.shape)

In [ ]:
print(sequence_indices[0][1])

In [ ]:
print(df.iloc[0])
print(df.iloc[12196])

In [55]:
print(sequence_indices[1][1])

[12197, 12198, 12199, 12200, 12201, 12202, 12203, 12204, 12205, 12206, 12207, 12208, 12209, 12210, 12211, 12212, 12213, 12214, 12215, 12216, 12217, 12218, 12219, 12220, 12221, 12222, 12223, 12224, 12225, 12226, 12227, 12228, 12229, 12230, 12231, 12232, 12233, 12234, 12235, 12236, 12237, 12238, 12239, 12240, 12241, 12242, 12243, 12244, 12245, 12246, 12247, 12248, 12249, 12250, 12251, 12252, 12253, 12254, 12255, 12256, 12257, 12258, 12259, 12260, 12261, 12262, 12263, 12264, 12265, 12266, 12267, 12268, 12269, 12270, 12271, 12272, 12273, 12274, 12275, 12276, 12277, 12278, 12279, 12280, 12281, 12282, 12283, 12284, 12285, 12286, 12287, 12288, 12289, 12290, 12291, 12292, 12293, 12294, 12295, 12296, 12297, 12298, 12299, 12300, 12301, 12302, 12303, 12304, 12305, 12306, 12307, 12308, 12309, 12310, 12311, 12312, 12313, 12314, 12315, 12316, 12317, 12318, 12319, 12320, 12321, 12322, 12323, 12324, 12325, 12326, 12327, 12328, 12329, 12330, 12331, 12332, 12333, 12334, 12335, 12336, 12337, 12338, 12339

In [56]:

print(df.iloc[12197])
print(df.iloc[17589])

Unnamed: 0                                                            12197
timestamp                                     2023-03-16 14:22:53.904183808
layers.sll.sll.pkttype                                                    4
layers.sll.sll.hatype                                                     1
layers.sll.sll.unused                                                 68:00
                                                ...                        
subscribers_count                                                       1.0
publishers_count                                                        1.0
msg_type                                                             Packet
msg_data                  <class 'theora_image_transport.msg._packet.Pac...
attack                                                                    1
Name: 12197, Length: 63, dtype: object
Unnamed: 0                                                            17589
timestamp                                     202

In [57]:
print(len(sequence_indices))

2347


In [58]:
from datetime import datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
tot_delta = 0

print(tot_delta)
for block in sequence_indices:
    first = df.iloc[block[1][0]]
    last = df.iloc[block[1][len(block[1]) - 1]]
    print(last['timestamp']-first['timestamp'])
    delta = last['timestamp']-first['timestamp']
    tot_delta = tot_delta + delta.total_seconds()
    print(tot_delta)

0
0 days 00:00:29.968583936
29.968583
0 days 00:00:24.659021824
54.627604
0 days 00:00:29.995751936
84.623355
0 days 00:00:23.024443136
107.64779800000001
0 days 00:00:30.006166528
137.653964
0 days 00:00:21.876401920
159.53036500000002
0 days 00:00:29.952431872
189.482796
0 days 00:00:22.087529984
211.570325
0 days 00:00:29.928455936
241.49878
0 days 00:00:22.943530240
264.44231
0 days 00:00:29.972819968
294.41512900000004
0 days 00:00:21.047045888
315.46217400000006
0 days 00:00:29.767504640
345.22967800000004
0 days 00:00:20.664383744
365.894061
0 days 00:00:30.000857856
395.894918
0 days 00:00:20.954331648
416.84924900000004
0 days 00:00:29.972609280
446.821858
0 days 00:00:22.809736704
469.631594
0 days 00:00:30.000954368
499.632548
0 days 00:00:23.073699840
522.706247
0 days 00:00:29.887668736
552.5939149999999
0 days 00:00:21.116257792
573.710172
0 days 00:00:29.751553536
603.4617249999999
0 days 00:00:20.724747264
624.1864719999999
0 days 00:00:30.001614848
654.1880859999999
0 

In [59]:
tot_delta/len(sequence_indices)

3294.0901547162275

### Convert the Pandas Dataframe to numpy

In [ ]:
Y=df['attack'].to_numpy()

In [ ]:
df = df.drop('attack', axis=1)

In [ ]:
print(df['timestamp'].dtype)

In [ ]:
timestamp = df['timestamp']

In [ ]:
print(timestamp)

In [ ]:
df['timestamp'] = pd.Categorical(df['timestamp'])
df['timestamp'] = df['timestamp'].cat.codes

In [ ]:
X=df.to_numpy()

In [ ]:
print(Y.shape)
print(X.shape)

In [ ]:
len(sequence_indices)

In [ ]:
x_total=[]
y_total=[]
time_total=[]
multi_total=[]

is_atk = sequence_indices[0][0]

for i in range(0, len(sequence_indices)): 
    #print(X[sequence_indices[i][1]].shape)
    #print(Y[sequence_indices[i][1]].shape)
    #print(timestamp[sequence_indices[i][1]].shape)
    if is_atk == sequence_indices[i][0]:
        continue
        
    x_data_block=np.vstack([X[sequence_indices[i-1][1]], X[sequence_indices[i][1]]])
    y_data_block=np.vstack([Y[sequence_indices[i-1][1], None], Y[sequence_indices[i][1], None]])
    #print(x_data_block)
    #print(timestamp[sequence_indices[i][1]])
    #print(timestamp[sequence_indices[i-1][1]])
    time_data_block=np.vstack([timestamp[sequence_indices[i-1][1], None], timestamp[sequence_indices[i][1], None]])
    multi_lab_block=np.vstack([mutli_atk_label[sequence_indices[i-1][1], None], mutli_atk_label[sequence_indices[i][1], None]])
    #print(x_data_block.shape)
    #print(y_data_block.shape)
    #print(time_data_block.shape)
    print(time_data_block)
    x_total.extend([x_data_block])    
    y_total.extend([y_data_block])
    time_total.extend([time_data_block])
    multi_total.extend([multi_lab_block])
    is_atk == sequence_indices[i][0]

In [ ]:
print(len(time_total))

In [ ]:
len(x_total)

In [ ]:
x_total[0]

### Train / Test Split

In [ ]:
#xtrain and xtest split
import random
PERCENTAGE_SPLIT=0.6

random_values=int(np.round(PERCENTAGE_SPLIT * len(y_total)))
indexes=random.sample(range(0, len(x_total)),random_values)

x_train=[x_total[i] for i in indexes]
y_train=[y_total[i] for i in indexes]
time_train=[time_total[i] for i in indexes]
multi_train=[multi_total[i] for i in indexes]

x_test=[x_total[i] for i in range(0, len(x_total)) if i not in indexes]
y_test=[y_total[i] for i in range(0, len(x_total)) if i not in indexes]
time_test=[time_total[i] for i in range(0, len(x_total)) if i not in indexes]
multi_test=[multi_total[i] for i in range(0, len(x_total)) if i not in indexes]

In [ ]:
len(x_train), len(y_train), len(time_train)

In [ ]:
#e ora mi rimane da rimettere tutto in fila per bene
X_train_sequence=np.vstack(x_train)
X_test_sequence=np.vstack(x_test)
y_train_sequence=np.vstack(y_train)
y_test_sequence=np.vstack(y_test)
time_test_sequence=np.vstack(time_test)
time_train_sequence=np.vstack(time_train)

In [ ]:
y_train_sequence.shape, X_train_sequence.shape, time_train_sequence.shape

In [ ]:
y_train_sequence.reshape(y_train_sequence.shape[0],)

In [ ]:
print(y_train_sequence.shape)

In [ ]:
y_train_sequence.shape, X_train_sequence.shape, X_test_sequence.shape, y_test_sequence.shape

In [ ]:
np.unique(y_train_sequence, return_counts=True), np.unique(y_test_sequence, return_counts=True)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
bst = XGBClassifier(n_estimators=10, max_depth=20, learning_rate=1.0, objective='binary:logistic')
# fit model
bst.fit(X_train_sequence, y_train_sequence)
# make predictions
preds = bst.predict(X_test_sequence)

In [ ]:
import sklearn
from sklearn import metrics
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

In [ ]:
acc=sklearn.metrics.accuracy_score(y_test_sequence, preds)
tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_test_sequence, preds).ravel()
print("usual accuracy and confusion matrix tn, fp, fn, tp")
acc, tn, fp, fn, tp

In [ ]:
fposr = fp / (tp + fp)
print(fposr)

In [ ]:
plt.figure(dpi=400)
fpr, tpr, _ = metrics.roc_curve(y_test_sequence,  preds)
plt.plot(fpr,tpr)
plt.ylabel('Recall')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
pr = sklearn.metrics.precision_score(y_test_sequence, preds)
rec = sklearn.metrics.recall_score(y_test_sequence, preds)
print(pr, rec)

In [ ]:
precision, recall, thr = metrics.precision_recall_curve(y_test_sequence,  preds)

In [ ]:
plt.figure(dpi=400)
plt.plot(recall, precision)
plt.ylabel('Precision')
plt.xlabel('Recall ')
plt.show()

In [ ]:
print("TRUE NEGATIVE RATE tn/(tn+fp)")
tn/(tn+fp)

In [ ]:
print("FALSE NEGATIVE RATE fn/(fn+tp)")
fn/(fn+tp)

In [ ]:
i=0
while(i < y_test_sequence.shape[0]):
    figure(figsize=(30, 2), dpi=400)
    plt.plot(y_test_sequence[i:i+300000])
    plt.show()
    i=i+300000

In [ ]:
i=0
while(i < y_test_sequence.shape[0]):
    figure(figsize=(15, 2), dpi=200)
    plt.plot(y_test_sequence[i:i+100000])
    plt.show()
    i=i+100000

In [ ]:
i=0
while(i < preds.shape[0]):
    figure(figsize=(15, 2), dpi=40)
    plt.plot(preds[i:i+300000])
    plt.show()
    i=i+300000

In [ ]:
i=0
while(i < preds.shape[0]):
    figure(figsize=(15, 2), dpi=200)
    plt.plot(preds[i:i+100000])
    plt.show()
    i=i+100000

In [ ]:
np.squeeze(y_test[0]).shape

In [ ]:
len(multi_test)

In [ ]:
multi_test[1][1]

In [ ]:
TARGET_FPR=0.0001
from dateutil.parser import parse


df_out = pd.DataFrame(columns=['start_idx_attack', 'end_idx_attack', 'attack_duration', 'time_to_detect', 'idx_detection_abs',
                               'idx_detection_rel', 'attack_len', 'attack_type','fpr','tpr','pr','rec','tn', 'fp','fn','tp'])

for i in range(0, len(x_test)):
    
    preds=bst.predict(x_test[i])
    preds_proba=bst.predict_proba(x_test[i])
    
    y_test_atk= np.where(y_test[i] == 1)[0]
    y_test_norm= np.where(y_test[i] == 0)[0]
    
    fpr, tpr, thresholds = metrics.roc_curve(y_test[i],  preds_proba[:,1])    
    fpr_index=np.min(np.where(fpr >= TARGET_FPR))-1
    fpr_value=fpr[fpr_index]
    tpr_value=tpr[fpr_index]
    
    preds = np.array([1 if prob > thresholds[fpr_index] else 0 for prob in preds_proba[:, 1]])
    
    y_temp = y_test[i].reshape((y_test[i].shape[0],))
    time_temp = time_test[i].reshape((time_test[i].shape[0],))
    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(np.squeeze(y_test[i]), preds).ravel()
    
    pr = sklearn.metrics.precision_score(y_test[i], preds)
    rec = sklearn.metrics.recall_score(y_test[i], preds)
    
    preds_1=preds[y_test_atk]
    
    date_time_0 = time_temp[y_test_atk[0]]
    date_time_last = time_temp[y_test_atk[y_test_atk.shape[0]-1]]
    
    date_time_0 = (date_time_0 - np.datetime64('1970-01-01T00:00:00Z'))/ np.timedelta64(1, 's')
    date_time_last = (date_time_last - np.datetime64('1970-01-01T00:00:00Z'))/ np.timedelta64(1, 's')
    attack_time = date_time_last - date_time_0
    
    if 1 in preds_1:
        index_rel=np.where(preds_1 == 1)[0][0]
        index= y_test_atk[index_rel]
        date_time_index = time_temp[index]
        date_time_index = (date_time_index - np.datetime64('1970-01-01T00:00:00Z'))/ np.timedelta64(1, 's')
        detection_time = date_time_index - date_time_0
    else:
        index_rel = -1
        index = -1
        detection_time = -1
    
    df_out.loc[i] = pd.Series({'start_idx_attack': y_test_atk[0], 'end_idx_attack': y_test_atk[y_test_atk.shape[0]-1],
                               'attack_duration': attack_time , 'time_to_detect': detection_time, 'idx_detection_abs': index,
                               'idx_detection_rel':index_rel, 'attack_len': preds_1.shape[0],
                               'attack_type':multi_test[i][y_test_atk[0]][0],'fpr':fpr_value,'tpr': tpr_value, 'pr': pr, 'rec': rec,
                               'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp})
    
    print("tpr {} , fpr {}".format(tpr_value, fpr_value))    
    print("the first detected 1 of a 1-series is at position: {}".format(index))

In [ ]:
df_out

In [ ]:
df_out.to_csv('./results_time_2.csv')

In [ ]:
import numpy as np

union_list = [
    'Tcp_Listen', 'layers.tcp.tcp.flags_tree.tcp.flags.ack', 'layers.tcp.tcp.stream', 
    'layers.ip.ip.checksum.status', 'layers.sll.sll.hatype', 'layers.tcp.tcp.analysis.tcp.analysis.acks_frame', 
    'Disk_Read', 'layers.ip.ip.flags_tree.ip.flags.rb', 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.severity', 
    'Cached', 'publishers_count', 'SwapFree', 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message', 
    'pgdeactivate', 'Tcp_Close', 'layers.tcp.tcp.window_size', 'Buffers', 'Active', 'layers.ip.ip.checksum', 
    'layers.tcp.tcp.options_tree.tcp.options.nop_tree.tcp.option_kind', 'pgactivate', 'Inactive', 'msg_type', 
    'Net_Received', 'layers.ssl.ssl.record.ssl.record.content_type', 'layers.icmpv6.icmpv6.type', 
    'layers.tcp.tcp.window_size_value', 'layers.tcp.tcp.payload', 'Net_Sent', 'pgfault', 
    'layers.tcp.tcp.flags_tree.tcp.flags.syn', 'Tcp_Syn', 'layers.ip.ip.version', 'subscribers_count', 
    'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt', 'layers.tcp.tcp.options_tree.tcp.options.nop', 
    'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.group', 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn', 
    'layers.tcp.tcp.ack', 'layers.tcp.tcp.options', 'nr_active_file', 'layers.sll.sll.unused', 
    'layers.sll.sll.pkttype', 'nr_inactive_file', 'pgfree', 'msg_data', 'pgpgin', 'Disk_Write', 
    'pgalloc_dma', 'MemFree', 'Tcp_Established', 'layers.ip.ip.flags_tree.ip.flags.df', 'pgmajfault', 
    'layers.ip.ip.flags', 'pgpgout', 'Tcp_TimeWait', 'layers.ipv6.ipv6.tclass_tree.ipv6.tclass.ecn', 
    'layers._ws.short', 'src_topic', 'layers.ipv6.ip.version'
]

# 轉成 numpy array (雖然 np.save 可以直接存 list，但轉成 array 比較保險)
union_array = np.array(union_list)

# 為了確保內容正確，先檢查長度是否為 60
print(f"Features count: {len(union_array)}")

if len(union_array) == 60:
    # 儲存檔案
    save_path = './data/usable_features.npy'
    np.save(save_path, union_array, allow_pickle=True)
    print(f"成功生成檔案: {save_path}")
    
    # 驗證讀取
    loaded_data = np.load(save_path, allow_pickle=True)
    print("驗證讀取成功，第一項資料為:", loaded_data[0])
else:
    print("長度不正確，請檢查列表內容。")